In [1]:
import os
import sys
import subprocess
import filecmp
import time
import difflib
import re
import hashlib

#讲解一下这个文件是干什么用的
#遍历test_folder下的所有.sy文件，调用你写的编译器前端，对每个测试文件生成.ll文件
#然后会用lli-14运行该.ll文件，如果有.in输入文件会将其作为输入，将输出与.out文件对比
#注意一定要让main.cpp重定向输出到.ll文件里面(不能有除了IR生成之外其他的调试日志)，需要改动launch.json中的目标文件路径

compiler_path = "./build_middle/compiler_middle_end"
test_folder = "./testcases"
ll_root = "./temp_worker/ll_out"
os.makedirs(ll_root, exist_ok=True)

# 颜色定义
class Color:
    GREEN = '\033[92m'      # 绿色 - AC通过
    RED = '\033[91m'        # 红色 - 错误
    YELLOW = '\033[93m'     # 黄色 - 超时
    RESET = '\033[0m'       # 重置颜色

# grab all the testcases

#递归地查找所有.sy文件
test_list = []
for root, dirs, files in os.walk(test_folder):
    for file in files:
        if file.endswith(".sy"):
            test_list.append(os.path.join(root, file))


def ll_path_for_test(test_path):
    rel = os.path.relpath(test_path, test_folder)
    digest = hashlib.md5(rel.encode("utf-8")).hexdigest()[:8]
    safe_rel = rel.replace("/", "__")
    return os.path.join(ll_root, f"{safe_rel}.{digest}.ll")


CE_list = []
LLI_FAIL_list = []
WA_list = []
AC_list = []
TLE_list = []
Time_Out = []
total_time = 0
Bad_test = []
Incomplete_input = []  # 新增：输入不完整的测试

def format_filename(path, max_len=25):
    """格式化文件名，保证长度一致"""
    filename = os.path.basename(path)
    if len(filename) > max_len:
        filename = filename[:max_len-3] + "..."
    return filename.ljust(max_len)

test_count = 0
for test in test_list:
    if test.endswith(".sy"):
        test_count += 1
        ll_file = ll_path_for_test(test)  # 生成.ll文件到临时目录，避免路径相关CE
        compile_args = [compiler_path, "-S", "-o", ll_file, test]
        try:
            ret = subprocess.run(compile_args, timeout=180)
        except subprocess.TimeoutExpired:
            Time_Out.append(test)
            formatted_name = format_filename(test)
            print(f"{Color.YELLOW}[TLE]{Color.RESET} {formatted_name} | Runtime:0.000000s | Total:{total_time:>6.2f}s | [{test_count}/{len(test_list)}]")
            continue
        if ret.returncode != 0:
            CE_list.append(test)
            formatted_name = format_filename(test)
            print(f"{Color.RED}[CE]{Color.RESET}  {formatted_name} | Runtime:0.000000s | Total:{total_time:>6.2f}s | [{test_count}/{len(test_list)}]")
            continue
        # run lli
        out_file = test[:-2] + "out"

        if not os.path.exists(test[:-2] + "in") and not os.path.exists(out_file):
            Bad_test.append(test)
            continue

        # Kill program run over 120s, give it a TLE
        try:
            if not os.path.exists(test[:-2] + "in"):
                ret = subprocess.run(["lli-14", "-opaque-pointers", ll_file], stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=120)
            else:
                ret = subprocess.run(["lli-14", "-opaque-pointers", ll_file], stdout=subprocess.PIPE, stderr=subprocess.PIPE, input=open(test[:-2] + "in").read().encode(), timeout=120)
        except subprocess.TimeoutExpired:
            TLE_list.append(test)
            formatted_name = format_filename(test)
            print(f"{Color.YELLOW}[TLE]{Color.RESET} {formatted_name} | Runtime:0.000000s | Total:{total_time:>6.2f}s | [{test_count}/{len(test_list)}]")
            total_time += 120
            continue

        #timer set
        runtime_time = 0
        try:
            if not os.path.exists(test[:-2] + "in"):
                ret = subprocess.run(["lli-14", "-opaque-pointers", ll_file], stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=120)
            else:
                with open(test[:-2] + "in") as input_file:
                    input_data = input_file.read().encode()
                    ret = subprocess.run(["lli-14", "-opaque-pointers", ll_file], stdout=subprocess.PIPE, stderr=subprocess.PIPE, input=input_data, timeout=120)

            stderr_output = ret.stderr.decode()
            match = re.search(r'TOTAL: (\d+)H-(\d+)M-(\d+)S-(\d+)us', stderr_output)
            if match:
                hours, minutes, seconds, microseconds = map(int, match.groups())
                # 将时间转换为秒
                runtime_time = hours * 3600 + minutes * 60 + seconds + microseconds / 1e6
                total_time += runtime_time
            else:
                # 如果没有找到TOTAL时间，使用0
                runtime_time = 0
        except subprocess.TimeoutExpired:
            TLE_list.append(test)
            formatted_name = format_filename(test)
            print(f"{Color.YELLOW}[TLE]{Color.RESET} {formatted_name} | Runtime:0.000000s | Total:{total_time:>6.2f}s | [{test_count}/{len(test_list)}]")
            total_time += 120
            continue

        # compare the output
        # Merge The reture Value and stdout

        dump_str = ret.stdout.decode()
        if dump_str and not dump_str.endswith('\n'):
            dump_str += "\n"
        dump_str += str(ret.returncode) + "\n"
        std_output = open(out_file).read()
        diff = difflib.unified_diff(dump_str.splitlines(), std_output.splitlines(), lineterm='')
        if len(list(diff)) != 0:
            WA_list.append(test)
            formatted_name = format_filename(test)
            print(f"{Color.RED}[WA]{Color.RESET}  {formatted_name} | Runtime:{runtime_time:.6f}s | Total:{total_time:>6.2f}s | [{test_count}/{len(test_list)}]")
            continue

        AC_list.append(test)
        formatted_name = format_filename(test)
        print(f"{Color.GREEN}[AC]{Color.RESET}  {formatted_name} | Runtime:{runtime_time:.6f}s | Total:{total_time:>6.2f}s | [{test_count}/{len(test_list)}]")


print()
print("=" * 60)
print(f"Compiler Error:  {len(CE_list):>3}")
print(f"Runtime Error:   {len(LLI_FAIL_list):>3}")
print(f"Timeout Error:   {len(TLE_list):>3}")
print(f"Wrong Answer:    {len(WA_list):>3}")
print(f"Bad Test:        {len(Bad_test):>3}")
print(f"{Color.GREEN}Accepted:        {len(AC_list):>3}{Color.RESET}")
print("=" * 60)
print(f"Total time: {total_time:.2f}s")
print()

# 检查失败的测试是否只有已知的7个存在输入缺陷的测试
known_defective_tests = {
    "2024-51P-30.sy",
    "2024-V6O-23.sy",
    "2024-AZ7-32.sy",
    "arm_if-combine2.sy",
    "arm_if-combine3.sy",
    "rv_if-combine2.sy",
    "rv_if-combine3.sy"
}

actual_failing_tests = {os.path.basename(test) for test in WA_list}

if actual_failing_tests == known_defective_tests:
    print(f"{Color.YELLOW}【检测结果】{Color.RESET}")
    print(f"失败的测试与已知的7个测试完全匹配")
    print(f"这些失败属于 {Color.YELLOW}lli-14 测试差异{Color.RESET}，不是编译器问题")
    print(f"根本原因：输入文件缺少数据（代码调用getint()2次，但输入仅1行）")
elif len(actual_failing_tests) > len(known_defective_tests):
    print(f"{Color.RED}【警告】编译器存在问题{Color.RESET}")
    print(f"除了已知的7个缺陷测试外，还有其他失败的测试:")
    extra_tests = actual_failing_tests - known_defective_tests
    for test in sorted(extra_tests):
        print(f"  - {test}")
elif len(actual_failing_tests) < len(known_defective_tests):
    print(f"{Color.GREEN}【好消息】{Color.RESET}")
    print(f"失败的测试数少于预期，某些缺陷测试已修复")
    fixed_tests = known_defective_tests - actual_failing_tests
    for test in sorted(fixed_tests):
        print(f"  ✓ {test} (已修复)")
else:
    print(f"失败的测试类型与预期不符（数量相同但内容不同）")
    print(f"实际失败: {sorted(actual_failing_tests)}")
    print(f"预期失败: {sorted(known_defective_tests)}")

[AC]  00_comment2.sy            | Runtime:0.000000s | Total:  0.00s | [1/382]
[AC]  34_multi_loop.sy          | Runtime:0.000000s | Total:  0.00s | [2/382]
[AC]  36_rotate.sy              | Runtime:0.000000s | Total:  0.00s | [3/382]
[AC]  38_light2d.sy             | Runtime:0.000000s | Total:  0.00s | [4/382]
[AC]  25_scope3.sy              | Runtime:0.000000s | Total:  0.00s | [5/382]
[AC]  37_dct.sy                 | Runtime:0.000000s | Total:  0.00s | [6/382]
[AC]  2024-DCV-7.sy             | Runtime:0.347726s | Total:  0.35s | [7/382]
[AC]  2024-K0I-17.sy            | Runtime:2.396787s | Total:  2.74s | [8/382]
[AC]  2024-7UJ-12.sy            | Runtime:0.425374s | Total:  3.17s | [9/382]
[AC]  2024-VTF-5.sy             | Runtime:1.613568s | Total:  4.78s | [10/382]
[AC]  2024-NKK-22.sy            | Runtime:2.010322s | Total:  6.79s | [11/382]
[AC]  2024-BAV-3.sy             | Runtime:17.159560s | Total: 23.95s | [12/382]
[AC]  2024-MXP-16.sy            | Runtime:2.138361s | Total:

In [2]:
print('CE', len(CE_list))
print('RE', len(LLI_FAIL_list))
print('TLE', len(TLE_list))
print('WA', len(WA_list))
print('AC', len(AC_list))
print('WA_LIST')
for t in sorted(WA_list):
    print(t)

CE 0
RE 0
TLE 0
WA 7
AC 375
WA_LIST
./testcases/final/performance/2024-51P-30.sy
./testcases/final/performance/2024-AZ7-32.sy
./testcases/final/performance/2024-V6O-23.sy
./testcases/performance_arm_25/arm_if-combine2.sy
./testcases/performance_arm_25/arm_if-combine3.sy
./testcases/performance_rv_25/rv_if-combine2.sy
./testcases/performance_rv_25/rv_if-combine3.sy


In [3]:
with open('/home/lyjy/LYJYCompiler/Compiler/temp_worker/wa_cases.txt', 'w', encoding='utf-8') as f:
    f.write(f'CE {len(CE_list)}\nRE {len(LLI_FAIL_list)}\nTLE {len(TLE_list)}\nWA {len(WA_list)}\nAC {len(AC_list)}\n')
    for t in sorted(WA_list):
        f.write(t + '\n')
print('wrote wa_cases.txt')

wrote wa_cases.txt


In [4]:
print('compile_timeout_count(Time_Out)=', len(Time_Out))
print('runtime_timeout_count(TLE_list)=', len(TLE_list))
print('ce_count=', len(CE_list), 'wa_count=', len(WA_list), 'ac_count=', len(AC_list))
print('\nfirst 20 runtime TLE cases:')
for t in TLE_list[:20]:
    print(t)
print('\nfirst 20 compile timeout cases:')
for t in Time_Out[:20]:
    print(t)


compile_timeout_count(Time_Out)= 0
runtime_timeout_count(TLE_list)= 0
ce_count= 0 wa_count= 7 ac_count= 375

first 20 runtime TLE cases:

first 20 compile timeout cases:
